# CSE5280 — Evacuation Simulation (Three-Floor Building)

**Author:** Joshua Cajuste
**Course:** CSE5280
**Instructor:** Eraldo Ribeiro
**Institution:** Florida Institute of Technology — Dept. of Electrical Engineering and Computer Science

---

## Overview

This notebook builds a multi-particle evacuation simulation of a three-floor building. All agent motion comes purely from minimizing a scalar cost function using gradient descent — no collision detection, no path planning, and no hard-coded floor switching. Everything emerges from the cost landscape.

The simulation covers:
- 20 agents randomly placed across three floors
- Two ground-floor exits with soft-min selection (agents naturally split)
- Two ramps for floor transitions (one per floor pair)
- Interior walls and obstacles forming a realistic floor layout
- Five cost terms: goal attraction, wall penalty, height adherence, smoothness, and agent repulsion
- Offscreen 3D rendering via `vedo` with frame-by-frame video export

## Table of Contents

1. [Installation & Setup](#1-installation--setup)
2. [Imports & Configuration](#2-imports--configuration)
3. [Simulation Parameters](#3-simulation-parameters)
4. [Building Geometry](#4-building-geometry)
   - 4.1 [Exits](#41-exits)
   - 4.2 [Ramps](#42-ramps)
   - 4.3 [Walls & Obstacles](#43-walls--obstacles)
5. [Geometry Helpers](#5-geometry-helpers)
6. [Ramp & Surface Height](#6-ramp--surface-height)
7. [Cost Function](#7-cost-function)
   - 7.1 [Goal Attraction (Soft-Min)](#71-goal-attraction--soft-min)
   - 7.2 [Wall Penalty](#72-wall-penalty)
   - 7.3 [Height-Adherence Cost](#73-height-adherence-cost)
   - 7.4 [Smoothness Term](#74-smoothness-term)
   - 7.5 [Agent Repulsion](#75-agent-repulsion)
   - 7.6 [Total Cost](#76-total-cost)
8. [Gradient Computation](#8-gradient-computation)
9. [Agent Initialization](#9-agent-initialization)
10. [3D Building Visualization](#10-3d-building-visualization)
11. [Simulation & Video Export](#11-simulation--video-export)
12. [Summary & Discussion](#12-summary--discussion)

## 1. Installation & Setup

Install required packages. Run this cell once before anything else.

In [19]:
!pip install -q vedo matplotlib imageio[ffmpeg]


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports & Configuration

We use `vedo` for 3D rendering with offscreen mode, which works in both VS Code notebooks and Colab without requiring a GUI backend or trame.

In [20]:
import numpy as np
from vedo import *

# Offscreen rendering - works everywhere (VS Code, Colab, no popup needed)
settings.default_backend = "vtk"


## 3. Simulation Parameters

All tunable constants are defined here in one place. Adjusting these changes both the physics and the visualization.

| Parameter | Symbol | Value | Description |
|-----------|--------|-------|-------------|
| Floor height | $H$ | 10 | Vertical gap between floors |
| Agents | $N$ | 20 | Number of simulated particles |
| Step size | $\alpha$ | 0.02 | Gradient descent learning rate |
| Soft-min temp | $\tau$ | 1.0 | Exit selection sharpness |
| Height stiffness | $w_h$ | 10 | Surface adherence weight |
| Smoothness weight | $w_s$ | 0.5 | Trajectory regularization weight |
| Repulsion radius | — | 1.0 | Personal space radius |
| Repulsion strength | — | 3.0 | Agent-agent repulsion weight |
| Max step | — | 0.5 | Step-size cap per iteration |
| Exit tolerance | — | 0.8 | XY radius to count as evacuated |
| Steps | — | 600 | Total simulation iterations |

In [21]:
# ── Simulation parameters ──────────────────────────────────────────────────
H                 = 10       # floor height
num_agents        = 25      # number of agents
alpha             = 0.02     # gradient descent step size
tau               = 1.0      # soft-min temperature
wh                = 60.0     # height-adherence weight
ws                = 0.5      # smoothness weight
repulsion_radius  = 1.0      # personal space radius
repulsion_strength= 3.0      # repulsion penalty weight
max_step          = 0.5      # step-size cap
exit_tol          = 0.8      # distance to consider agent "evacuated"
exit_radius       = 1.0      # radius of the exit area
steps             = 600      # total simulation iterations

## 4. Building Geometry

### 4.1 Exits

Two exits sit on the ground floor ($z = 0$), placed at opposite corners of the floor plan. This forces agents from upper floors to travel diagonally across the building before descending. Exit selection is handled entirely by the soft-min cost — no agent is ever hard-assigned to a specific exit.

In [22]:
# ── Exits (ground floor only) ──────────────────────────────────────────────
exit1 = np.array([2.0,  2.0,  0.0])
exit2 = np.array([18.0, 18.0, 0.0])
exits = [exit1, exit2]

### 4.2 Ramps

Each ramp is a capsule-shaped corridor in the floor plan with a linearly interpolated height function. Two ramps are defined:

- **Ramp 1**: connects Floor 1 ($z = H$) down to the Ground floor ($z = 0$) — runs north–south through the center
- **Ramp 2**: connects Floor 2 ($z = 2H$) down to Floor 1 ($z = H$) — runs east–west through the center

The `width` parameter defines the capsule half-radius — agents within this distance of the centerline are considered on the ramp.

In [23]:
# ── Ramps ──────────────────────────────────────────────────────────────────
ramps = [
    {"A": np.array([10.0, 4.0]),  "B": np.array([10.0, 16.0]), "z0": H,   "z1": 0,   "width": 3.0},
    {"A": np.array([4.0,  10.0]), "B": np.array([16.0, 10.0]), "z0": 2*H, "z1": H,   "width": 3.0},
]


### 4.3 Walls & Obstacles

Walls are 2D line segments used by the wall cost to keep agents inside the building and navigating around barriers. This is an extension of the previous 2D floor-plan assignment, so the layout includes both outer boundary walls and interior obstacles per floor.

**Ground floor**: a central corridor with side rooms created by vertical dividers, plus a horizontal cross-bar obstacle forcing agents to route around it.

**Obstacle**: a box obstacle in the center of the ground floor forces agents to navigate around it rather than cutting straight to the exit. This was added per the assignment requirement to use layouts and obstacles.

In [ ]:
# ── Walls & Obstacles (2D line segments) ──────────────────────────────────
# Outer boundary
walls = [
    ((0,0),   (20,0)),
    ((20,0),  (20,20)),
    ((20,20), (0,20)),
    ((0,20),  (0,0)),

    # Interior dividers — ground floor corridor layout
    ((5,0),   (5,8)),
    ((5,12),  (5,20)),
    ((15,0),  (15,8)),
    ((15,12), (15,20)),

    # Horizontal cross-bar obstacle — forces agents to route around center
    ((7,9),   (13,9)),
    ((7,11),  (13,11)),

    # Box obstacle near center of ground floor
    ((8,13),  (12,13)),
    ((8,17),  (12,17)),
    ((8,13),  (8,17)),
    ((12,13), (12,17)),
]


## 5. Geometry Helpers

#@title ### Point-to-Segment Distance
#

This helper is used by both the wall cost and ramp detection. For a point $P$ and segment $[A, B]$:

$$t = \text{clip}\!\left(\frac{(P-A)\cdot(B-A)}{\|B-A\|^2},\; 0,\; 1\right), \qquad d(P,[A,B]) = \|P - (A + t(B-A))\|$$

In [25]:
def point_segment_distance(P, A, B):
    """Euclidean distance from point P to segment [A, B]."""
    v = B - A
    t = np.dot(P - A, v) / np.dot(v, v)
    t = np.clip(t, 0, 1)
    return np.linalg.norm(P - (A + t * v))

## 6. Ramp & Surface Height

### 6.1 Ramp Height Function

For a ramp with centerline $[A, B]$ and heights $z_0$ (top) and $z_1$ (bottom), the height at any $(x, y)$ inside the footprint is:

$$z_{\text{ramp}}(x,y) = z_0 + (z_1 - z_0)\, u(x,y), \qquad u = \text{clip}\!\left(\frac{(P-A)\cdot(B-A)}{\|B-A\|^2}, 0, 1\right)$$

### 6.2 Surface Height

Outside any ramp, the surface is a flat floor. The full piecewise surface is:

$$z_{\text{surf}}(x,y,z) = \begin{cases} z_{\text{ramp}}(x,y) & \text{if } (x,y) \in \mathcal{R} \\ \max\{f \in \{0, H, 2H\} : f \leq z + 0.5\} & \text{otherwise} \end{cases}$$

The floor-snapping always picks the **highest floor the agent is currently standing on**, preventing agents from being pulled up to the ceiling of the floor below.

In [ ]:
def ramp_height(x, y, ramp):
    """Linear height interpolation along ramp centerline."""
    A, B = ramp["A"], ramp["B"]
    P = np.array([x, y])
    v = B - A
    u = np.clip(np.dot(P - A, v) / np.dot(v, v), 0, 1)
    return ramp["z0"] + (ramp["z1"] - ramp["z0"]) * u


def surface_height(x, y, z):
    """
    Target surface z for a particle at (x, y, z).
    Inside a ramp footprint: use interpolated ramp height.
    Elsewhere: snap to the highest floor level <= z + 0.5
    (always snaps DOWN — prevents agents hovering above exits on a lower floor).
    """
    P = np.array([x, y])
    for ramp in ramps:
        dist = point_segment_distance(P, ramp["A"], ramp["B"])
        if dist <= ramp["width"]:
            return ramp_height(x, y, ramp)
    floors = [0.0, float(H), float(2 * H)]
    valid  = [f for f in floors if f <= z + 0.5]
    return max(valid) if valid else 0.0


## 7. Cost Function

The total cost driving every agent is:

$$C(p) = C_{\text{goal}} + C_{\text{walls}} + C_{\text{height}} + C_{\text{smooth}} + C_{\text{repulsion}}$$

Each term enforces a different geometric or physical constraint. No term ever hard-codes behavior — everything is emergent from the gradient.

#@title ### Cost Function Implementations
#

### 7.1 Goal Attraction — Soft-Min

A hard minimum is non-differentiable, so we use the soft-min formulation:

$$C_{\text{goal}}(p) = -\tau \log \sum_{i=1}^{2} \exp\!\left(-\frac{\|p_{xy} - p_i^{\text{exit}}\|^2}{\tau}\right)$$

Note: we use **XY-only distance** to the exits. This keeps the $z$-gradient of $C_{\text{goal}}$ zero — height is handled entirely by $C_{\text{height}}$, preventing the two terms from fighting each other near the exit.

The gradient is a weighted combination of individual exit attractions:

$$\nabla C_{\text{goal}} = \sum_i w_i \nabla C_i, \qquad w_i = \frac{\exp(-C_i/\tau)}{\sum_j \exp(-C_j/\tau)}$$

Temperature $\tau = 1.0$ gives a smooth blend; lower $\tau$ makes agents commit more sharply to one exit.

In [ ]:
def goal_cost(p):
    """
    Soft-min attraction to exits using XY-only distance.
    XY-only ensures goal_cost has zero z-gradient — height adherence
    handles vertical placement, preventing the two terms from conflicting.
    """
    costs = np.array([np.linalg.norm(p[:2] - e[:2])**2 for e in exits])
    return -tau * np.log(np.sum(np.exp(-costs / tau)))


### 7.2 Wall Penalty

Quadratic band penalty — activates when an agent comes within radius $r = 1.0$ of any wall segment:

$$C_{\text{walls}} = \sum_k \max(0,\; r - d_k(p))^2$$

This is the same formulation as the previous 2D assignment, now applied in 3D using the XY projection of the agent position.

In [28]:
def wall_cost(p):
    """
    Quadratic band penalty: C_walls = sum_k max(0, r - d_k)^2
    """
    cost = 0.0
    r = 1.0
    for w in walls:
        A = np.array(w[0], dtype=float)
        B = np.array(w[1], dtype=float)
        d = point_segment_distance(p[:2], A, B)
        if d < r:
            cost += (r - d)**2
    return cost

### 7.3 Height-Adherence Cost

Keeps agents glued to the building surface (floor slabs and ramp inclines):

$$C_{\text{height}} = w_h\,(z - z_{\text{surf}}(x,y))^2$$

An extra underground penalty kicks in if $z < 0$ to prevent agents from sinking through the ground floor:

$$C_{\text{underground}} = 20\, w_h\, z^2 \quad \text{if } z < 0$$

In [ ]:
def height_cost(p):
    """
    Surface-adherence cost: penalises deviation from the building surface.
    Extra underground penalty prevents agents from phasing through the ground.
    """
    x, y, z = p
    z_surf = surface_height(x, y, z)
    cost = wh * (z - z_surf) ** 2
    if z < 0.0:
        cost += wh * 20.0 * z ** 2
    return cost


### 7.4 Smoothness Term

Temporal regularization that penalises abrupt jumps between steps:

$$C_{\text{smooth}} = w_s\,\|p_{k+1} - p_k\|^2$$

This reduces oscillation, especially near walls and ramp edges where gradients can be large.

In [30]:
def smoothness_cost(p, p_prev):
    """
    Trajectory smoothness: C_smooth = ws * ||p - p_prev||^2
    """
    return ws * np.linalg.norm(p - p_prev)**2

### 7.5 Agent Repulsion

Short-range repulsion prevents agents from occupying the same space:

$$C_{\text{repulsion}} = \sum_{j \neq i} \varphi(\|p_i - p_j\|), \qquad \varphi(d) = s \cdot \max(0,\, r_{\text{rep}} - d)^2$$

Same family as the wall penalty — quadratic band — so the gradient magnitude scales smoothly with overlap depth.

In [31]:
def repulsion_cost(p, agents):
    """
    Short-range quadratic repulsion from all other agents.
    """
    cost = 0.0
    for a in agents:
        d = np.linalg.norm(p - a)
        if 0 < d < repulsion_radius:
            cost += repulsion_strength * (repulsion_radius - d)**2
    return cost

### 7.6 Total Cost

$$C(p) = C_{\text{goal}} + C_{\text{walls}} + C_{\text{height}} + C_{\text{smooth}} + C_{\text{repulsion}}$$

In [32]:
def total_cost(p, p_prev, agents):
    return (
        goal_cost(p)
        + wall_cost(p)
        + height_cost(p)
        + smoothness_cost(p, p_prev)
        + repulsion_cost(p, agents)
    )

## 8. Gradient Computation

#@title ### Finite-Difference Gradient
#

Gradients are computed numerically using central finite differences:

$$\frac{\partial C}{\partial p_i} \approx \frac{C(p + \epsilon\, e_i) - C(p - \epsilon\, e_i)}{2\epsilon}, \qquad \epsilon = 10^{-3}$$

This avoids deriving analytical gradients for each cost term, which is especially convenient for the piecewise surface height function.

In [33]:
def gradient(p, p_prev, agents, eps=1e-3):
    """Finite-difference gradient of total_cost w.r.t. p."""
    g = np.zeros(3)
    for i in range(3):
        dp = np.zeros(3)
        dp[i] = eps
        g[i] = (total_cost(p + dp, p_prev, agents)
               - total_cost(p - dp, p_prev, agents)) / (2 * eps)
    return g

## 9. Agent Initialization

Agents are seeded with a fixed random state (`seed=42`) for reproducibility. Each agent gets a random $(x,y)$ position within the floor bounds and is placed on one of the three floors uniformly at random.

In [34]:
np.random.seed(42)

agents = []
for _ in range(num_agents):
    x     = np.random.uniform(2, 18)
    y     = np.random.uniform(2, 18)
    floor = np.random.choice([0, H, 2*H])
    agents.append(np.array([x, y, float(floor)]))

agents      = np.array(agents)
prev_agents = agents.copy()          # needed for smoothness term
active      = np.ones(num_agents, dtype=bool)  # False when agent exits

print(f"Initialized {num_agents} agents across floors 0, {H}, {2*H}")

Initialized 25 agents across floors 0, 10, 20


## 10. 3D Building Visualization

#@title ### Building Scene Construction
#

The 3D scene is built to match the stacked cutaway isometric style from the assignment figure:

- **Floor slabs**: thick concrete-coloured boxes, one per floor, clearly separated by a gap
- **Outer walls**: south and west faces are full-height; north and east are cut short so the interior is visible from the camera angle
- **Interior walls and obstacles**: short dividers per floor, plus the box obstacle on the ground floor
- **Ramps**: oriented inclined quads with solid side cheeks dropping to the lower floor level
- **Exit markers**: green door frames with a cone arrow and glow sphere
- **Agents**: red spheres that turn grey when evacuated

In [35]:

import numpy as np
from vedo import *

# ═══════════════════════════════════════════════════════════════════
# DETAILED 3-FLOOR BUILDING VISUALIZATION
# Inspired by the stacked cutaway isometric style in the assignment
# ═══════════════════════════════════════════════════════════════════

W   = 20.0   # building width/depth
T   = 0.25   # wall thickness
FH  = H - 1  # usable floor-to-ceiling height (leave 1 unit gap between floors)

all_objects = []

# ───────────────────────────────────────────────────────────────────
# Helper: flat panel mesh from corner points
# ───────────────────────────────────────────────────────────────────
def panel(x0,y0,z0, x1,y1,z1, x2,y2,z2, x3,y3,z3, color="white", alpha=0.8):
    pts   = [[x0,y0,z0],[x1,y1,z1],[x2,y2,z2],[x3,y3,z3]]
    faces = [[0,1,2,3]]
    return Mesh([pts,faces]).color(color).alpha(alpha).lw(0)

def box_mesh(cx,cy,cz, sx,sy,sz, color="white", alpha=0.85):
    return Box(pos=(cx,cy,cz), size=(sx,sy,sz)).color(color).alpha(alpha)

# ───────────────────────────────────────────────────────────────────
# FLOOR SLABS  (thick, with a warm concrete colour)
# ───────────────────────────────────────────────────────────────────
SLAB = 0.4
for k, col in enumerate(["#d4c5a9","#cfc0a3","#c8b89a"]):
    z = k * H
    all_objects.append(box_mesh(W/2, W/2, z - SLAB/2, W+T*2, W+T*2, SLAB, col, 0.95))

# ───────────────────────────────────────────────────────────────────
# OUTER WALLS  — only South & West are full-height (visible faces)
#               North & East are cut low so interior is visible
# ───────────────────────────────────────────────────────────────────
def outer_walls_for_floor(z_base, full_h, cut_h, wall_col="#f0ece4", alpha=0.82):
    walls = []
    # South wall  (y = 0) — full height
    walls.append(box_mesh(W/2, 0,    z_base+full_h/2, W+T*2, T, full_h, wall_col, alpha))
    # West wall   (x = 0) — full height
    walls.append(box_mesh(0,   W/2,  z_base+full_h/2, T, W,   full_h, wall_col, alpha))
    # North wall  (y = W) — cut short so we can see inside
    walls.append(box_mesh(W/2, W,    z_base+cut_h/2,  W+T*2, T, cut_h, wall_col, alpha*0.7))
    # East wall   (x = W) — cut short
    walls.append(box_mesh(W,   W/2,  z_base+cut_h/2,  T, W,   cut_h, wall_col, alpha*0.7))
    return walls

for k in range(3):
    z = k * H
    all_objects += outer_walls_for_floor(z, FH, FH*0.35)

# ───────────────────────────────────────────────────────────────────
# INTERIOR WALLS  — short dividers that add layout complexity
# Same layout on every floor (slightly different per floor)
# ───────────────────────────────────────────────────────────────────
def interior_wall(x0,y0,x1,y1, z_base, height, col="#e8dfd0", alpha=0.80):
    A = np.array([x0,y0],dtype=float)
    B = np.array([x1,y1],dtype=float)
    mid = (A+B)/2
    L   = np.linalg.norm(B-A)
    ang = np.degrees(np.arctan2(y1-y0, x1-x0))
    b   = Box(pos=(mid[0], mid[1], z_base+height/2), size=(L, T, height))
    b.rotate_z(ang)
    return b.color(col).alpha(alpha)

# Ground floor
z=0; h=FH*0.6
all_objects += [
    interior_wall(5,  0,  5,  8,  z, h),
    interior_wall(5,  12, 5,  W,  z, h),
    interior_wall(15, 0,  15, 8,  z, h),
    interior_wall(15, 12, 15, W,  z, h),
    interior_wall(5,  8,  9,  8,  z, h*0.7),
    interior_wall(11, 8,  15, 8,  z, h*0.7),
    interior_wall(8,  12, 12, 12, z, h*0.7),
]

# First floor
z=H; h=FH*0.55
all_objects += [
    interior_wall(7,  0,  7,  9,  z, h),
    interior_wall(7,  13, 7,  W,  z, h),
    interior_wall(13, 0,  13, 9,  z, h),
    interior_wall(13, 13, 13, W,  z, h),
    interior_wall(7,  9,  13, 9,  z, h*0.7),
]

# Second floor
z=2*H; h=FH*0.5
all_objects += [
    interior_wall(6,  0,  6,  10, z, h),
    interior_wall(6,  14, 6,  W,  z, h),
    interior_wall(14, 5,  14, W,  z, h),
    interior_wall(6,  10, 10, 10, z, h*0.65),
    interior_wall(10, 14, 10, W,  z, h*0.65),
]

# ───────────────────────────────────────────────────────────────────
# RAMP SURFACES  — properly oriented inclined quads
# ───────────────────────────────────────────────────────────────────
def make_ramp_mesh(ramp, col="#8B6914"):
    A2 = ramp["A"]; B2 = ramp["B"]
    z0 = ramp["z0"]; z1 = ramp["z1"]; w = ramp["width"]
    dx = B2[0]-A2[0]; dy = B2[1]-A2[1]
    L  = np.sqrt(dx*dx+dy*dy)
    px, py = -dy/L*w, dx/L*w
    pts = [
        [A2[0]+px, A2[1]+py, z0],
        [A2[0]-px, A2[1]-py, z0],
        [B2[0]-px, B2[1]-py, z1],
        [B2[0]+px, B2[1]+py, z1],
    ]
    faces = [[0,1,2,3]]
    m = Mesh([pts,faces]).color(col).alpha(0.92).lw(1)
    # Side panels to make the ramp look like a solid ramp
    side1 = Mesh([[ pts[0], pts[3],
                    [pts[3][0],pts[3][1],z1-H],
                    [pts[0][0],pts[0][1],z0-H]], [[0,1,2,3]]]).color("#6b5210").alpha(0.85)
    side2 = Mesh([[ pts[1], pts[2],
                    [pts[2][0],pts[2][1],z1-H],
                    [pts[1][0],pts[1][1],z0-H]], [[0,1,2,3]]]).color("#6b5210").alpha(0.85)
    return [m, side1, side2]

for r in ramps:
    all_objects += make_ramp_mesh(r)

# ───────────────────────────────────────────────────────────────────
# EXIT MARKERS  — green archway + glow sphere
# ───────────────────────────────────────────────────────────────────
for ex in exits:
    # Door opening (green rectangle on wall)
    all_objects.append(box_mesh(ex[0], ex[1], 1.2, 1.8, 0.15, 2.4, "#00ff88", 0.95))
    # Glow sphere above
    all_objects.append(Sphere(pos=(ex[0], ex[1], 2.8), r=0.55).color("#00ff44").alpha(0.9))

# ───────────────────────────────────────────────────────────────────
# AGENT SPHERES
# ───────────────────────────────────────────────────────────────────
agent_spheres = [Sphere(a, r=0.38).color("tomato").alpha(1.0) for a in agents]
all_objects  += agent_spheres

print("Detailed scene built:", len(all_objects), "objects")


Detailed scene built: 67 objects


## 11. Simulation & Video Export

Each step of the simulation:

1. For each active agent, compute the finite-difference gradient of the total cost
2. Apply the gradient descent update: $p \leftarrow p - \alpha \nabla C$
3. Cap the step size to `max_step = 0.5` to prevent numerical blowup
4. Deactivate agents whose XY position is within `exit_tol` of any exit and $z < H/2$
5. Capture a screenshot every 2 steps

After the loop, all frames are assembled into an MP4 at 12 fps using `imageio`.

In [36]:
# ── Plotter (offscreen) ───────────────────────────────────────────────────
plt = Plotter(bg="#1a1a2e", bg2="#16213e", title="Building Evacuation",
              offscreen=True, size=(1280, 720))

plt.show(all_objects, interactive=False,
         camera=dict(
             pos=(60, -40, 52),
             focal_point=(10, 10, 12),
             viewup=(0, 0, 1)
         ))

# ── Simulation + frame capture ─────────────────────────────────────────────
import os, imageio
from IPython.display import Video, display

os.makedirs("frames", exist_ok=True)
frames_paths = []

for step in range(steps):
    if not np.any(active):
        print(f"All agents evacuated at step {step}!")
        break

    new_agents = agents.copy()

    for i, p in enumerate(agents):
        if not active[i]:
            continue

        for e in exits:
            # XY distance only — agent deactivates when directly above exit
            if np.linalg.norm(p[:2] - e[:2]) < exit_tol and p[2] < H * 0.5:
                active[i] = False
                agent_spheres[i].color("grey").alpha(0.25)
                break

        if not active[i]:
            continue

        others = agents[np.arange(num_agents) != i]
        g      = gradient(p, prev_agents[i], others)
        new_p  = p - alpha * g

        step_vec  = new_p - p
        step_size = np.linalg.norm(step_vec)
        if step_size > max_step:
            step_vec = step_vec / step_size * max_step
            new_p    = p + step_vec

        new_agents[i] = new_p
        agent_spheres[i].pos(new_p)

    prev_agents = agents.copy()
    agents      = new_agents

    if step % 3 == 0:
        path = f"frames/frame_{step:04d}.png"
        plt.screenshot(path)
        frames_paths.append(path)

    if step % 50 == 0:
        print(f"Step {step:3d} | Evacuated: {int(np.sum(~active))}/{num_agents}")

plt.close()

# ── Assemble video ─────────────────────────────────────────────────────────
video_path = "evacuation.mp4"
with imageio.get_writer(video_path, fps=20, codec="libx264",
                        ffmpeg_params=["-pix_fmt", "yuv420p"]) as writer:
    for fp in frames_paths:
        writer.append_data(imageio.imread(fp))

print("\nVideo saved:", video_path)
display(Video(video_path, embed=True, width=700))


Step   0 | Evacuated: 0/25
Step  50 | Evacuated: 3/25
Step 100 | Evacuated: 5/25
Step 150 | Evacuated: 6/25
Step 200 | Evacuated: 6/25
Step 250 | Evacuated: 6/25
Step 300 | Evacuated: 6/25
Step 350 | Evacuated: 6/25
Step 400 | Evacuated: 6/25
Step 450 | Evacuated: 6/25
Step 500 | Evacuated: 6/25
Step 550 | Evacuated: 6/25


C:\Users\cajus\AppData\Local\Temp\ipykernel_50964\1593027469.py:71: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  writer.append_data(imageio.imread(fp))



Video saved: evacuation.mp4


## 12. Summary & Discussion

### Cost Function Design

| Term | Formula | Role |
|------|---------|------|
| $C_{\text{goal}}$ | Soft-min, XY-only | Attracts agents to exits; enables natural group splitting |
| $C_{\text{walls}}$ | Quadratic band | Keeps agents inside corridors and away from obstacles |
| $C_{\text{height}}$ | Quadratic + underground clamp | Constrains agents to floors/ramps; enables floor transitions |
| $C_{\text{smooth}}$ | Squared step size | Stabilises trajectories; reduces oscillation near boundaries |
| $C_{\text{repulsion}}$ | Quadratic band | Models crowd interaction; prevents agent overlap |

### Design Decisions

**XY-only goal cost**: using full 3D distance caused the goal gradient's z-component to fight the height cost near exits, leaving agents stuck orbiting the exit at a fixed height. Separating responsibilities — goal handles XY navigation, height cost handles z — eliminates this conflict entirely.

**Floor-snapping direction**: using `int(z/H)` (truncate down) instead of `round(z/H)` means agents always snap to the floor they're standing on, not the ceiling above. Combined with the valid-floor-list approach, this prevents agents from floating halfway between floors.

**Ramp width**: increasing the ramp capture radius to 3.0 gives agents a wider corridor to step onto, which is important because gradient descent takes small steps and a narrow ramp can be missed entirely.

**Obstacles and layout**: the interior walls and center box obstacle on the ground floor are a direct extension of the 2D floor-plan assignment. They make the evacuation non-trivial — agents can't cut straight to the exit and must navigate around barriers, which makes the emergent routing behavior more visible.

### Observations

Agents on upper floors move toward the ramp entrance before descending — this routing is entirely emergent from the height-adherence cost. No rule tells them to go to the ramp; the cost landscape simply makes standing on the ramp cheaper than hovering in mid-air between floors.

The two exits attract roughly equal numbers of agents when agents start uniformly distributed, with natural clustering near each exit based on starting position. Agents near the box obstacle visibly route around it before heading to the exit.